# **GROUP 2 – NATURAL LANGUAGE PROCESSING**


---


## **"Implementation of a Guardrail for Detecting Mixed-Language Hate Speech on Indonesian Social Media"**


---



### **Group Members:**
### 1) Michael Respati Sanjaya Ho - 535250008
### 2) Felix Lin - 535250044
### 3) Charlie Jhodya Soe - 535250180


---



## **Tarumanagara University**





## **A. Preparation and Load Model**

**CONNECT TO GOOGLE DRIVE**

Google Drive is used to store and retrieve
fine-tuned models so that the models do not need to be
re-trained every time the notebook is used.


In [20]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**CHECK THE TRAINING OUTPUT MODEL**

Ensure that the fine-tuned model previously
saved to Google Drive can be located and used.

In [21]:
import os

model_path = '/content/drive/MyDrive/HateGuard/model/model_hate_speech_final'

print("Models found:", os.path.exists(model_path))
print("Contents of the ‘model’ folder:")

for item in os.listdir(model_path):
    print(item)

Models found: True
Contents of the ‘model’ folder:
tokenizer.json
model.safetensors
tokenizer_config.json
config.json


**LIBRARY INSTALLATION**

Install the libraries required to load
and run the Transformer model.

In [22]:
!pip install transformers torch -q

**IMPORT LIBRARY**

Import libraries to:
- run PyTorch
- calculate the probability of prediction outcomes
- load the tokeniser and Transformer model

In [23]:
import re
import torch
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

print("The library has been successfully loaded.")

The library has been successfully loaded.


**LOAD MODEL AND TOKENISER**

Loading the tokeniser and the fine-tuned model from
Google Drive.

The model is not retrained in this notebook.
The model is used solely for prediction/inference.

In [24]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("The model and tokeniser have been successfully loaded.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

The model and tokeniser have been successfully loaded.


**SELECTING THE DEVICE**

Select whether the model will run on the
GPU or the CPU.

The GPU is used, if available, so that the prediction process
can be carried out more quickly.

In [25]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

model.to(device)
model.eval()

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


**LOAD ALAY DICTIONARY**

Loads the same Alay dictionary as that used
during the pre-processing of the training data.

The aim is to ensure that the pre-processing of new comments is consistent
with the pre-processing used when the model was trained.


In [26]:
import pandas as pd

url_dictionary = (
    "https://raw.githubusercontent.com/"
    "okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection/"
    "master/new_kamusalay.csv"
)

alay_dictionary = pd.read_csv(
    url_dictionary,
    encoding='latin-1',
    header=None,
    names=['alay', 'baku']
)

kamus_dict = dict(
    zip(
        alay_dictionary['alay'],
        alay_dictionary['baku']
    )
)

print("Number of words in the dictionary:", len(kamus_dict))

Number of words in the dictionary: 15167


## **B. Preprocessing Functions**

**TEXT PREPROCESSING**

Perform the same preprocessing as during training:
1. Case folding
2. Remove user placeholders and URLs
3. Split the text into words
4. Normalise slang into standard words

In [27]:
# Text cleaning functions
def clean_text(text):
    """Performing case folding and normalisation of 'alay' language."""
    text = str(text).lower()                             # case folding
    text = text.replace('user', '').replace('url', '')   # clear the crawler placeholder
    words = text.split()
    words = [kamus_dict.get(w, w) for w in words]        # standardisation of slang -> standard language
    return ' '.join(words)

## **C. PREDICTION**

**PROBABILITY PREDICTIONS AND GUARDRAIL LEVELS**

 Converting the model’s output into probabilities using Softmax.

 p_toxic: the probability that the text constitutes hate speech.

 This probability is then used as the basis
 for determining the guardrail severity level:
 safe / weak / moderate / strong.


In [28]:
import torch.nn.functional as F
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
# Converting the model’s output into probabilities for the 'safe' and 'toxic' classes.

def toxic_prediction(text):
    """Return a confidence score (0–1) indicating whether this text is toxic."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=-1)
    return probs[0][1].item()

def prob_to_level(prob_toxic: float) -> str:
    """Convert the confidence score to a severity level for the guardrail (section 9)."""
    if prob_toxic < 0.50:
        return 'secure'
    elif prob_toxic < 0.75:
        return 'weak'
    elif prob_toxic < 0.90:
        return 'moderate'
    else:
        return 'strong'

# Example
example = "sia teh belegug, bagong sia teh"
skor = toxic_prediction(example)
print(f"Text: {example}")
print(f"Confidence toxic: {skor:.2%} -> Level: {prob_to_level(skor)}")

Text: sia teh belegug, bagong sia teh
Confidence toxic: 99.87% -> Level: strong


## **D. GUARDRAIL**

**GUARDRAIL ACTION RULES**

 Determines system actions based on the level:

 safe     → comments are displayed as normal
 weak     → comments are displayed and flagged
 moderate → comments are displayed and flagged
 strong   → comments are blocked and escalated

 This function defines the system’s action rules,
 not the model training process.

In [29]:
def guardrail_measures(level: str) -> dict:
    """Level map -> system action"""
    rules = {
        'secure':     {'action': 'tayang_normal', 'display': True,  'escalation': False},
        'weak':     {'action': 'flag_log',      'display': True,  'escalation': False},
        'moderate': {'action': 'flag_log',      'display': True,  'escalation': False},
        'strong':   {'action': 'blokir',        'display': False, 'escalation': True},
    }
    return rules[level]

for lv in ['secure', 'weak', 'moderate', 'strong']:
    print(lv, '->', guardrail_measures(lv))


secure -> {'action': 'tayang_normal', 'display': True, 'escalation': False}
weak -> {'action': 'flag_log', 'display': True, 'escalation': False}
moderate -> {'action': 'flag_log', 'display': True, 'escalation': False}
strong -> {'action': 'blokir', 'display': False, 'escalation': True}


**MODERATOR QUEUE**

 Holds comments requiring human review.

 Only comments with an ‘escalation’ value of ‘True’ are placed
 in the moderator queue.

 Under the current rules, only the ‘STRONG’ level is placed
 in the moderator queue.

In [30]:
moderator_queue = []

def process_comment(text, predicted_level):
    '''Processes a comment and adds a “strong” level entry to the moderator queue'''
    action = guardrail_measures(predicted_level)
    if action['escalation']:
        moderator_queue.append({
            "text": text,
            "level": predicted_level,
            "status": "awaiting_review"
        })
    return action

**END-TO-END COMMENT PREDICTION AND GUARDRAIL**

 This function takes a single new comment and performs the following:

 1. pre-processing
 2. tokenisation
 3. IndoBERTweet prediction
 4. calculation of safe/toxic probability
 5. determination of severity level
 6. determination of guardrail action
 7. adding strong comments to the moderator queue

 The output is returned in the form of a dictionary

In [31]:
import torch
import torch.nn.functional as F

def comment_prediction(text):
    """Performing pre-processing, predicting hate speech, determining severity levels, and implementing guardrail measures."""
    # Use the same pre-processing as for the training data
    text_clean = clean_text(text)

    # Tokenization
    inputs = tokenizer(
        text_clean,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move to device model
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Predictions
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    # Convert logits to probabilities
    probabilities = F.softmax(outputs.logits, dim=-1)
    p_secure = probabilities[0][0].item()
    p_toxic = probabilities[0][1].item()

    # Specify the label
    if p_toxic < 0.50:
        level = "secure"
    elif p_toxic < 0.75:
        level = "weak"
    elif p_toxic < 0.90:
        level = "moderate"
    else:
        level = "strong"

    # Follow the guardrail rules
    action = process_comment(text, level)

    return {
        "text": text,
        "text_clean": text_clean,
        "p_secure": p_secure,
        "p_toxic": p_toxic,
        "level": level,
        "action": action
    }


## **E. TESTING**

**COMMENT PREDICTION TEST**

An example of a manual test to examine the prediction results
of IndoBERTweet and the guardrail measures.


In [32]:
result = comment_prediction("You look like you’re under a lot of stress")

print(result)

{'text': 'You look like you’re under a lot of stress', 'text_clean': 'kamu look like you’re under a lot of stres', 'p_secure': 0.913000226020813, 'p_toxic': 0.08699973672628403, 'level': 'secure', 'action': {'action': 'tayang_normal', 'display': True, 'escalation': False}}


**CHECK THE MODERATOR QUEUE**

Displays all comments currently in
the moderator queue.


In [33]:
# Display comments currently in the moderator queue
print(" MODERATOR QUEUE ")

for item in moderator_queue:
    print(item)

 MODERATOR QUEUE 


**RESET MODERATOR QUEUE**

This cell is only used to clear test results
before carrying out a new demonstration.


In [34]:
# Clearing the moderator queue before the next test
moderator_queue = []

print("Moderator queue:", moderator_queue)

Moderator queue: []


**END-TO-END ANALYSIS FUNCTION**

Encapsulates comment predictions into a single function.

This function takes the prediction results and simplifies them
into information that can be easily used by the UI.


In [35]:
def comment_analysis(text):
    """Generate end-to-end comment analysis results."""
    result = comment_prediction(text)

    level = result['level']
    p_toxic = result['p_toxic']
    action = result['action']

    return {
        'text': text,
        'teks_clean': result['text_clean'],
        'level': level,
        'confidence': p_toxic,
        'Behavior': action
    }


**MODERATION OUTPUT FORMAT**

Displays the analysis results in a format that is easier
for humans to read:
- input
- text after pre-processing
- level
- confidence
- action

This function could form the basis for a web UI.

In [36]:
def show_analysis(text):
    result = comment_analysis(text)

    print(" MODERATION RESULT ")
    print(f"Input      : {result['text']}")
    print(f"Clean text : {result['teks_clean']}")
    print(f"Prediksi   : {result['level'].upper()}")
    print(f"Confidence : {result['confidence'] * 100:.2f}%")

    if result['Behavior']['action'] == 'tayang_normal':
        print("Measure   : Normal Broadcasting")
    elif result['Behavior']['action'] == 'flag_log':
        print("Measure   : TAGGED / LOG IN")
    elif result['Behavior']['action'] == 'blokir':
        print("Measure   : BLOCKED + SENT TO THE MODERATOR’S QUEUE")


**EXAMPLE OF SYSTEM TESTING**

Testing the system using a single example sentence to
check whether the pipeline:

input → pre-processing → prediction → level → action
is functioning correctly.


In [37]:
show_analysis("Jhodya mahasiswa tertolol")

 MODERATION RESULT 
Input      : Jhodya mahasiswa tertolol
Clean text : jhodya mahasiswa tertolol
Prediksi   : STRONG
Confidence : 99.62%
Measure   : BLOCKED + SENT TO THE MODERATOR’S QUEUE


## **F.USER INTERFACE**

## **G. Model Comparison**

To compare the performance of the fine-tuned Transformer model with the Gemini (zero-shot) model, we require comparison data. This data must include:

*   `y_true_compare`: The ground truth labels from the dataset used for comparison.
*   `y_pred_bert_compare`: Predictions from the Transformer model that has been fine-tuned on that dataset.
*   `y_pred_llm_compare`: Predictions from the Gemini (zero-shot) model on the same dataset.

As this data is not yet available in the notebook, I will create *placeholder* data so that the visualisation code can run. You must replace this with the actual data from your model comparison results.

In [38]:
import gradio as gr

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;600;700;800&display=swap');

body, .gradio-container {
    background: linear-gradient(135deg, #0b1121 0%, #172554 100%) !important;
    font-family: 'Plus Jakarta Sans', sans-serif !important;
    background-attachment: fixed;
}

/* Glassmorphism App Shell */
#app-shell {
    max-width: 1050px;
    margin: 0 auto;
    background: rgba(255, 255, 255, 0.03) !important;
    backdrop-filter: blur(16px);
    -webkit-backdrop-filter: blur(16px);
    border: 1px solid rgba(255, 255, 255, 0.1);
    border-radius: 24px;
    padding: 35px 30px 40px 30px;
    box-shadow: 0 30px 60px rgba(0,0,0,0.4);
}

/* Header Text Animasi Gradien */
#app-header {
    text-align: center;
    margin-bottom: 30px;
}
#app-header h1 {
    background: linear-gradient(to right, #60a5fa, #a855f7, #60a5fa);
    background-size: 200% auto;
    color: transparent !important;
    -webkit-background-clip: text;
    background-clip: text;
    font-weight: 800;
    font-size: 38px;
    margin-bottom: 8px;
    letter-spacing: -0.5px;
    animation: shine 4s linear infinite;
}
@keyframes shine {
    to { background-position: 200% center; }
}
#app-header p {
    color: #94a3b8 !important;
    font-size: 15px;
    font-weight: 400;
    margin: 0;
}

/* Animation Appears for the Card Panel */
@keyframes slideUpFade {
    from { opacity: 0; transform: translateY(20px); }
    to { opacity: 1; transform: translateY(0); }
}

.panel-card {
    background-color: #ffffff !important;
    border-radius: 16px;
    padding: 24px;
    box-shadow: 0 10px 25px rgba(0,0,0,0.15);
    border: 1px solid rgba(255,255,255,0.6);
    animation: slideUpFade 0.6s ease-out forwards;
    transition: transform 0.3s ease, box-shadow 0.3s ease;
}

/* Hover Effect When the Mouse is Over the Panel */
.panel-card:hover {
    transform: translateY(-4px);
    box-shadow: 0 20px 40px rgba(0,0,0,0.2);
}

/* Correct ALL label/title text to ensure it is high-contrast */
.panel-card *,
.panel-card label,
.panel-card span,
.panel-card p,
.panel-card h1,
.panel-card h2,
.panel-card h3,
.panel-card .section-label {
    color: #0f172a !important;
    font-weight: 700 !important;
}

.section-label {
    font-size: 16px !important;
    border-bottom: 2px solid #e2e8f0;
    padding-bottom: 8px;
    margin-bottom: 16px !important;
    color: #1e3a8a !important; /* Biru tua elegan */
}

/* Styling Input & Output Textbox with Focus Effect (Glow) */
textarea, input[type="text"] {
    background-color: #f8fafc !important;
    border: 1px solid #cbd5e1 !important;
    border-radius: 12px !important;
    color: #0f172a !important;
    font-weight: 600 !important;
    transition: all 0.3s ease !important;
}
textarea:focus, input[type="text"]:focus {
    border-color: #3b82f6 !important;
    box-shadow: 0 0 0 4px rgba(59, 130, 246, 0.2) !important;
    background-color: #ffffff !important;
}

/* Button Styling Examples (Example Comments) */
.panel-card button[class*="gallery-item"],
.panel-card [id*="component"] button {
    background-color: #ffffff !important;
    border: 1px solid #cbd5e1 !important;
    color: #3b82f6 !important;
    font-weight: 600 !important;
    border-radius: 8px !important;
    transition: all 0.2s ease;
}
.panel-card button[class*="gallery-item"]:hover {
    background-color: #eff6ff !important;
    border-color: #3b82f6 !important;
    transform: scale(1.02);
}

/* Styling Accordion */
#app-shell [class*="accordion"] {
    background-color: #f8fafc !important;
    border: 1px solid #cbd5e1 !important;
    border-radius: 12px !important;
    margin-top: 12px;
    transition: all 0.3s ease;
}
#app-shell [class*="accordion"]:hover {
    border-color: #94a3b8 !important;
}

/* MAIN BUTTON ANIMATION (Liquid Gradient) */
#analyze_btn {
    background: linear-gradient(45deg, #1d4ed8, #4f46e5, #1d4ed8) !important;
    background-size: 200% auto !important;
    color: #ffffff !important;
    font-weight: 800 !important;
    font-size: 15px !important;
    border-radius: 12px !important;
    border: none !important;
    box-shadow: 0 4px 15px rgba(79, 70, 229, 0.4) !important;
    transition: all 0.3s ease !important;
}
#analyze_btn:hover {
    background-position: right center !important;
    transform: translateY(-3px) scale(1.02) !important;
    box-shadow: 0 8px 25px rgba(79, 70, 229, 0.6) !important;
}
#analyze_btn:active {
    transform: translateY(1px) scale(0.98) !important; /* Efek ditekan */
}
#analyze_btn * {
    color: #ffffff !important;
}

/* Clear Button */
#clear_btn {
    background-color: #f1f5f9 !important;
    color: #475569 !important;
    font-weight: 700 !important;
    border-radius: 12px !important;
    border: 1px solid #cbd5e1 !important;
    transition: all 0.2s ease !important;
}
#clear_btn:hover {
    background-color: #e2e8f0 !important;
    color: #0f172a !important;
    transform: translateY(-2px);
}
#clear_btn * {
    color: inherit !important;
}

/* Moderator View Section */
#moderator-title {
    text-align: center;
    color: #ffffff !important;
    font-weight: 800 !important;
    margin-top: 40px !important;
    margin-bottom: 16px !important;
    font-size: 24px !important;
    text-shadow: 0 2px 10px rgba(0,0,0,0.5);
}

#moderator-card {
    background-color: #ffffff !important;
    border-radius: 16px;
    padding: 24px;
    box-shadow: 0 10px 30px rgba(0,0,0,0.3);
    animation: slideUpFade 0.8s ease-out forwards;
}
#moderator-card * {
    color: #0f172a !important;
}

#refresh_btn {
    background-color: #eff6ff !important;
    color: #1d4ed8 !important;
    font-weight: 800 !important;
    border-radius: 10px !important;
    border: 1px solid #bfdbfe !important;
    transition: all 0.2s ease !important;
}
#refresh_btn:hover {
    background-color: #dbeafe !important;
    transform: scale(1.01);
}

#status_box textarea {
    font-weight: 800 !important;
    color: #1d4ed8 !important;
    font-size: 15px !important;
}
"""

def run_analysis(text):
    """Called when the ‘Comment Analysis’ button is pressed."""
    if not text or not text.strip():
        return (
            "-", "-", "-",
            "⚠️ Please enter a comment first.",
            format_blocked_list(),
        )

    result = comment_analysis(text)  # Functions of a notebook (Section E)

    level_text = result["level"].upper()
    conf_text = f"{result['confidence'] * 100:.2f}%"
    preprocess_text = result["teks_clean"]

    action = result["Behavior"]["action"]
    if action == "tayang_normal":
        status_text = "✅ DISPLAYED NORMALLY"
    elif action == "flag_log":
        status_text = "⚠️ FLAGGED / SENT TO LOG"
    elif action == "blokir":
        status_text = "⛔ BLOCKED + SENT TO MODERATOR QUEUE"
    else:
        status_text = action

    return level_text, conf_text, preprocess_text, status_text, format_blocked_list()


def format_blocked_list():
    """Rewrite the contents of the 'moderator_queue' section to make it more readable."""
    if not moderator_queue:  # global variables from the notebook (Section D)
        return "No blocked comments at the moment."

    lines = []
    for i, item in enumerate(moderator_queue, start=1):
        lines.append(
            f"{i}. \"{item['text']}\"\n"
            f"   Level    : {item['level'].upper()}\n"
            f"   Status   : {item['status']}"
        )
    return "\n\n".join(lines)


def clear_all():
    """Called when the 'Clear' button is pressed."""
    return "", "-", "-", "-", "", format_blocked_list()


COMMENT_EXAMPLE = [
    "woi hitam tidur besok jadi budak",
    "semua orang cina harus dibunuh",
    "presiden tergoblok sepanjang sejarah",
    "jhodya mahasiswa tertolol",
]


with gr.Blocks(css=CUSTOM_CSS, title="HateGuard AI") as demo:
    with gr.Column(elem_id="app-shell"):

        gr.HTML(
            """
            <div id="app-header">
                <h1>HateGuard AI</h1>
                <p>Automated Moderation System Based on IndoBERT &amp; Guardrail Severity Levels</p>
            </div>
            """
        )

        with gr.Row():

            # LEFT-HAND COLUMN: Analysis of Public Comments
            with gr.Column(elem_classes="panel-card"):
                gr.Markdown("**Public Comment Analysis**", elem_classes="section-label")
                input_box = gr.Textbox(
                    placeholder="Type or paste the netizen's comment here.",
                    lines=5,
                    show_label=False,
                    elem_id="comments_box",
                )
                gr.Examples(
                    examples=COMMENT_EXAMPLE,
                    inputs=input_box,
                    label="Example comments (click to use)",
                )
                with gr.Row():
                    clear_btn = gr.Button("Clear", elem_id="clear_btn")
                    analyze_btn = gr.Button("Comment Analysis", elem_id="analyze_btn")

            # RIGHT-HAND COLUMN: System Decision Status
            with gr.Column(elem_classes="panel-card"):
                gr.Markdown("**Level Deteksi**", elem_classes="section-label")
                level_box = gr.Textbox(
                    value="-", interactive=False, show_label=False, elem_id="level_box"
                )

                gr.Markdown("**Confidence Toxic**", elem_classes="section-label")
                conf_box = gr.Textbox(
                    value="-", interactive=False, show_label=False, elem_id="conf_box"
                )

                gr.Markdown("**System Status**", elem_classes="section-label")
                status_box = gr.Textbox(
                    value="", interactive=False, show_label=False, elem_id="status_box"
                )

                with gr.Accordion("See the preprocessing process", open=False):
                    preprocess_box = gr.Textbox(
                        value="-",
                        interactive=False,
                        show_label=False,
                        elem_id="preprocess_box",
                        lines=2,
                    )

        # Moderator View
        gr.Markdown("## Moderator View", elem_id="moderator-title")

        with gr.Column(elem_id="moderator-card"):
            with gr.Accordion("Blocked comment queue (Action Required)", open=False):
                refresh_btn = gr.Button(
                    "🔄 Refresh Queue",
                    elem_id="refresh_btn",
                )
                blocked_box = gr.Textbox(
                    value="No blocked comments at the moment.",
                    interactive=False,
                    show_label=False,
                    lines=6,
                    elem_id="blocked_box",
                )

    analyze_btn.click(
        fn=run_analysis,
        inputs=input_box,
        outputs=[level_box, conf_box, preprocess_box, status_box, blocked_box],
    )

    clear_btn.click(
        fn=clear_all,
        inputs=None,
        outputs=[input_box, level_box, conf_box, preprocess_box, status_box, blocked_box],
    )

    refresh_btn.click(
        fn=format_blocked_list,
        inputs=None,
        outputs=blocked_box,
    )

demo.launch(share=True)

/tmp/ipykernel_536/3491785375.py:278: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="HateGuard AI") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://202ef95692d10bd9c0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
